# Ejercicio 1 – Embeddings contextualizados (BERT)
Tema 5 – BERT, vector medio y máximo, similitud coseno

Enunciado:
Dadas la frase de consulta
📌 "Dogs are domestic animals."
y las frases:
📌 ["Dogs are pets.", "This is a dog.", "They are free today."]
se pide:

Obtener el vector medio de cada frase usando bert-base-uncased.

Obtener el vector máximo de cada frase.

Calcular la similitud coseno entre la frase de consulta y cada frase.

Reflexionar: ¿cuál es más similar? ¿es mejor el vector medio o el máximo?

In [ ]:
import tqdm

In [38]:
# Cargar modelo y tokenizer de BERT
from transformers import BertTokenizer, TFBertModel
import tensorflow as tf
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

#Cargar tokenizer y modelo
tokenizer=BertTokenizer.from_pretrained("bert-base-uncased")
model= TFBertModel.from_pretrained("bert-base-uncased")

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

In [39]:
consulta = "Dogs are domestic animals."
frases = ["Dogs are pets.", "This is a dog.", "They are free today."]

In [40]:
def obtener_vector(texto, tipo='mean'):
    # Tokenizamos el texto y lo convertimos en tensores
    tokens = tokenizer(texto, return_tensors='tf', padding=True, truncation=True)
    
    # Pasamos por BERT
    salida = model(**tokens)
    
    # Obtenemos la última capa oculta
    embeddings = salida.last_hidden_state[0]  # shape: (num_tokens, 768)
    
    # Eliminamos [CLS] y [SEP] (primer y último token)
    embeddings = embeddings[1:-1]

    # Media o máximo
    if tipo == 'mean':
        return tf.reduce_mean(embeddings, axis=0).numpy()
    elif tipo == 'max':
        return tf.reduce_max(embeddings, axis=0).numpy()
        
# Obtener vectores
vector_consulta_mean = obtener_vector(consulta, tipo='mean')
vector_consulta_max = obtener_vector(consulta, tipo='max')

vectores_frases_mean = [obtener_vector(f, tipo='mean') for f in frases]
vectores_frases_max = [obtener_vector(f, tipo='max') for f in frases]

# Calcular similitudes de coseno
print("🔷 Similitud con vector medio:")
for frase, vec in zip(frases, vectores_frases_mean):
    sim = cosine_similarity([vector_consulta_mean], [vec])[0][0]
    print(f"- {frase} → {sim:.3f}")

print("\n🔶 Similitud con vector máximo:")
for frase, vec in zip(frases, vectores_frases_max):
    sim = cosine_similarity([vector_consulta_max], [vec])[0][0]
    print(f"- {frase} → {sim:.3f}")


🔷 Similitud con vector medio:
- Dogs are pets. → 0.817
- This is a dog. → 0.698
- They are free today. → 0.571

🔶 Similitud con vector máximo:
- Dogs are pets. → 0.798
- This is a dog. → 0.811
- They are free today. → 0.758


# Ejercicio 2: Fine-tuning de BERT con un dataset propio
Supongamos que tienes un conjunto de datos pequeño con frases etiquetadas como positivas (1) o negativas (0). El objetivo es ajustar (fine-tune) un modelo BERT para esta tarea de clasificación binaria.

In [12]:
import tensorflow as tf
from transformers import BertTokenizer, TFBertForSequenceClassification
from sklearn.model_selection import train_test_split
import numpy as np

sentences = [
    "I love this movie", "This film is great", "What a terrible film",
    "I hated the movie", "Fantastic story", "Worst movie ever"
]
labels = [1, 1, 0, 0, 1, 0]  # 1 = positivo, 0 = negativo

#tokenizar con BERT
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Convertimos las frases al formato que necesita BERT
inputs = tokenizer(
    sentences,
    padding=True,
    truncation=True
)


In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    inputs["input_ids"],
    np.array(labels),
    test_size=0.3,
    random_state=42
)

attention_train = inputs["attention_mask"][:len(X_train)]
attention_test = inputs["attention_mask"][len(X_train):]

# Convertir arrays a tensores
X_train_tensor = tf.convert_to_tensor(X_train)
X_test_tensor = tf.convert_to_tensor(X_test)

y_train_tensor = tf.convert_to_tensor(y_train)
y_test_tensor = tf.convert_to_tensor(y_test)

attention_train_tensor = tf.convert_to_tensor(attention_train)
attention_test_tensor = tf.convert_to_tensor(attention_test)


In [ ]:
model = TFBertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)


All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [16]:

loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metric = tf.keras.metrics.SparseCategoricalAccuracy("accuracy")

model.compile(optimizer='adam', loss=loss, metrics=[metric])


In [23]:
model.fit(
    [X_train_tensor, attention_train_tensor],
    y_train_tensor,
    validation_data=([X_test_tensor, attention_test_tensor], y_test_tensor),
    epochs=5,
    batch_size=2
)


Epoch 1/5
2/2 [==============================] - 2s 922ms/step - loss: 0.6324 - accuracy: 0.7500 - val_loss: 1.4613 - val_accuracy: 0.0000e+00
Epoch 2/5
2/2 [==============================] - 2s 894ms/step - loss: 0.8195 - accuracy: 0.2500 - val_loss: 0.6869 - val_accuracy: 1.0000
Epoch 3/5
2/2 [==============================] - 2s 908ms/step - loss: 0.6408 - accuracy: 0.7500 - val_loss: 1.1265 - val_accuracy: 0.0000e+00
Epoch 4/5
2/2 [==============================] - 2s 831ms/step - loss: 0.6614 - accuracy: 0.7500 - val_loss: 1.9233 - val_accuracy: 0.0000e+00
Epoch 5/5
2/2 [==============================] - 2s 842ms/step - loss: 0.6310 - accuracy: 0.7500 - val_loss: 2.1384 - val_accuracy: 0.0000e+00


In [25]:
loss, accuracy = model.evaluate([X_test_tensor, attention_test_tensor], y_test_tensor)
print(f"Accuracy: {accuracy:.2f}")


1/1 [==============================] - 0s 152ms/step - loss: 2.1384 - accuracy: 0.0000e+00
Accuracy: 0.00


# Ej 3: similitud semántica

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("paraphrase-MiniLM-L6-v2")
embedding = model.encode("Esta es una frase.") #coddifica frases


# representación en bow binario, ft


In [34]:
from sklearn.feature_extraction.text import CountVectorizer
corpus= ["El gato no gato maulla", "El perro ladra", "Maulla el gato"]

vectorizer_binary = CountVectorizer(binary=True)
X1 = vectorizer_binary.fit_transform(corpus)
X1=X1.toarray()
names=vectorizer_binary.get_feature_names_out()
print(names)
print(X1)



['el' 'gato' 'ladra' 'maulla' 'no' 'perro']
[[1 1 0 1 1 0]
 [1 0 1 0 0 1]
 [1 1 0 1 0 0]]


In [ ]:

vectorizer = CountVectorizer()
X2 = vectorizer.fit_transform(corpus)
X2=X2.toarray()
names2=vectorizer.get_feature_names_out()
print(names2)
print(X2)


['el' 'gato' 'ladra' 'maulla' 'no' 'perro']
[[1 2 0 1 1 0]
 [1 0 1 0 0 1]
 [1 1 0 1 0 0]]


In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer_tfidf = TfidfVectorizer()
X3 = vectorizer_tfidf.fit_transform(corpus)
X3=X3.toarray()
names3=vectorizer_tfidf.get_feature_names_out()
print(names3)
print(X3)


['el' 'gato' 'ladra' 'maulla' 'no' 'perro']
[[0.28680065 0.73861611 0.         0.36930805 0.48559571 0.        ]
 [0.38537163 0.         0.65249088 0.         0.         0.65249088]
 [0.48133417 0.61980538 0.         0.61980538 0.         0.        ]]


# Ej 4 NER Spacy

In [41]:
import spacy

# Cargar el modelo pequeño de inglés
nlp = spacy.load("en_core_web_sm")

# Frase de ejemplo
text = "Barack Obama was the 44th president of the United States."

# Procesar la frase
doc = nlp(text)

# Mostrar entidades detectadas
for ent in doc.ents:
    print(f"Texto: {ent.text} | Etiqueta: {ent.label_} | Explicación: {spacy.explain(ent.label_)}")


Texto: Barack Obama | Etiqueta: PERSON | Explicación: People, including fictional
Texto: 44th | Etiqueta: ORDINAL | Explicación: "first", "second", etc.
Texto: the United States | Etiqueta: GPE | Explicación: Countries, cities, states
